In [5]:
#! pip install chromadb
#! pip install -U langchain langchain-community langchain-openai pydantic


In [6]:
from config_loader import *
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

In [8]:
from pathlib import Path
Path('./download/10k_html/MSFT-20250730.pdf').stem

'MSFT-20250730'

In [36]:
loader=PyMuPDFLoader(
    file_path='download\\10k_pdf\\MSFT-20250730.pdf'
)
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
docs=loader.load()
split_docs=splitter.split_documents(docs)
persist_dir = "chroma_store_large"
embed=OpenAIEmbeddings(model=os.environ['OPENAI_EMBEDED_MODEL'])
vector_store=Chroma.from_documents(
    documents=split_docs,
    embedding=embed,
    persist_directory=persist_dir,
    collection_name="MSFT"
    )


Retriever

In [56]:
retriever= vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
    )
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=retriever.invoke(query)
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)



----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

In [57]:
retriever2= vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":2}
    )
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=retriever2.invoke(query)
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)



----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

In [69]:
retriever2= vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={'score_threshold': 0.1}
    
    )
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=retriever2.invoke(query)
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)



----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

In [65]:
result

[Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal li

In [48]:
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=vector_store.similarity_search(query,k=5)
result[0].page_content

'PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal lives. As with many innovations, AI presents \nrisks and challenges that could affect its adoption, and therefore our business. AI algorithms or training \nmethodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate \ninformation. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.'

In [51]:
query="identify sentiments around AI as a growth driver vs a controlled risk"
result=vector_store.similarity_search_with_score(query,k=5)
result

[(Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal l

Multi query retrievers

In [78]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

model=ChatOpenAI(
    model=os.environ["OPENAI_MODEL"],
    temperature=0,
    max_completion_tokens=100)

base_retriever=vector_store.as_retriever(search_type='similarity',search_kwargs={'k':2})


prompt=PromptTemplate(
    template="""Please provide the information as a top leader and remove all noises from the answers .
      Query is {query}""",
      input_variables=[query])

query="identify sentiments around AI as a growth driver vs a controlled risk"

retriver=MultiQueryRetriever.from_llm(
    llm=model,
    retriever=base_retriever,
    prompt=prompt
)
result=retriever.invoke(query)
print(result)


[Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='PART I\nItem 1A\nIssues in the development, deployment, and use of AI may result in reputational or competitive \nharm\n \nor\n \nliability. We are building AI into many of our offerings, including our productivity services, and we are also \nmaking AI available for our customers to use in solutions that they build. This AI may be developed by \nMicrosoft or others, including our strategic partner, OpenAI. We expect these elements of our business to \ngrow. We envision a future in which AI operating in devices, applications, and the cloud helps our  \ncustomers be more productive in their work and personal li

In [79]:

for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)


----Result 1-----

PART I
Item 1A
Issues in the development, deployment, and use of AI may result in reputational or competitive 
harm
 
or
 
liability. We are building AI into many of our offerings, including our productivity services, and we are also 
making AI available for our customers to use in solutions that they build. This AI may be developed by 
Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to 
grow. We envision a future in which AI operating in devices, applications, and the cloud helps our  
customers be more productive in their work and personal lives. As with many innovations, AI presents 
risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training 
methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate 
information. Content generated by AI systems may be offensive, illegal, inaccurate, or otherwise harmful.

----Result 2-

Contecxtual retreiver

In [89]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_openai import ChatOpenAI
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import PromptTemplate

model = ChatOpenAI(temperature=0)


retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 2}
)

compressor = LLMChainExtractor.from_llm(
    llm=model
)

contextual_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor
)

query = "identify sentiments around AI as a growth driver vs a controlled risk"
result = contextual_retriever.invoke(query)  # plain string

print(result)


[Document(metadata={'author': '', 'creationDate': '', 'creationdate': '', 'creator': '', 'file_path': 'download\\10k_pdf\\MSFT-20250730.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': '', 'moddate': '', 'page': 39, 'producer': 'WeasyPrint 66.0', 'source': 'download\\10k_pdf\\MSFT-20250730.pdf', 'subject': '', 'title': '10-K', 'total_pages': 167, 'trapped': ''}, page_content='We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business.')]


In [90]:
for i, doc in enumerate(result):
    print(f'\n----Result {i+1}-----\n')
    print(doc.page_content)


----Result 1-----

We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business.
